# CardioIA — Fase 2 — Estetoscópio digital

Notebook único com as duas partes do enunciado:

1. **Parte 1:** leitura de frases de sintomas + mapa de conhecimento (CSV) + sugestão de diagnóstico por regras.
2. **Parte 2:** classificação de **alto risco** / **baixo risco** com **TF-IDF** e modelos do scikit-learn (frases derivadas do dataset numérico da Fase 1).

### Google Colab

- Se precisar: `!pip install pandas scikit-learn`
- Coloque na mesma pasta (`datasets/` ou `/content/datasets`) os arquivos: `sintomas_pacientes.txt`, `mapa_conhecimento.csv`, `frases_risco.csv`.
- Opção: clonar o repositório e abrir este notebook; a célula abaixo procura `datasets` na raiz do projeto, em `../datasets` ou em `/content/datasets`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

candidatos = [
    Path("datasets"),
    Path("../datasets"),
    Path("/content/datasets"),
]
DATA = next(
    (p for p in candidatos if (p / "mapa_conhecimento.csv").exists()),
    Path("datasets"),
)
print("Pasta de dados:", DATA.resolve())

---
## Parte 1 — Frases, mapa de conhecimento e sugestão de diagnóstico

Arquivos: `sintomas_pacientes.txt` e `mapa_conhecimento.csv` (colunas `sintoma_1`, `sintoma_2`, `doenca_associada`).

In [ ]:
def sintoma_na_frase(sintoma: str, texto: str) -> bool:
    s = sintoma.lower().strip()
    t = texto.lower()
    if s in t:
        return True
    if s == "desmaio" and "desmai" in t:
        return True
    return False


def pontuar_linha(frase: str, sintoma_1: str, sintoma_2: str) -> int:
    m1 = sintoma_na_frase(sintoma_1, frase)
    m2 = sintoma_na_frase(sintoma_2, frase)
    if m1 and m2:
        return 2
    if m1 or m2:
        return 1
    return 0


def sugerir_diagnosticos(frase: str, mapa: pd.DataFrame) -> str:
    scores: dict[str, int] = {}
    pares_completos: dict[str, int] = {}
    for _, row in mapa.iterrows():
        p = pontuar_linha(frase, str(row["sintoma_1"]), str(row["sintoma_2"]))
        if p == 0:
            continue
        doenca = str(row["doenca_associada"])
        scores[doenca] = scores.get(doenca, 0) + p
        if p == 2:
            pares_completos[doenca] = pares_completos.get(doenca, 0) + 1
    if not scores:
        return "Nenhuma associação encontrada no mapa de conhecimento."
    ordenado = sorted(
        scores.items(),
        key=lambda x: (-pares_completos.get(x[0], 0), -x[1], x[0]),
    )
    principal = ordenado[0][0]
    detalhe = "; ".join(f"{d} ({v})" for d, v in ordenado[:5])
    return f"Sugestão principal: {principal}. Agregado: {detalhe}"

In [ ]:
mapa = pd.read_csv(DATA / "mapa_conhecimento.csv", encoding="utf-8")
with open(DATA / "sintomas_pacientes.txt", encoding="utf-8") as f:
    frases = [ln.strip() for ln in f if ln.strip()]

for i, frase in enumerate(frases, 1):
    print(f"\n--- Relato {i} ---")
    print(frase[:160] + ("..." if len(frase) > 160 else ""))
    print(sugerir_diagnosticos(frase, mapa))

### Parte 1 — Governança / limitações

Este protótipo **não** substitui avaliação médica: é simulação educacional. Regras fixas e texto livre podem gerar **falsos positivos/negativos**; em produção seriam necessários validação clínica, dados representativos e critérios transparentes.

---
## Parte 2 — TF-IDF e classificação de risco (triagem textual)

Arquivo: `frases_risco.csv` (`frase`, `situacao`). As frases foram construídas a partir do `dataset_cardiologia.csv` da Fase 1: **alto risco** quando `historico_doenca_cardiaca > 0`, **baixo risco** quando zero.

In [ ]:
df = pd.read_csv(DATA / "frases_risco.csv", encoding="utf-8")
df["situacao"] = df["situacao"].str.strip().str.lower()
print(df["situacao"].value_counts())
df.head(3)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["frase"],
    df["situacao"],
    test_size=0.25,
    random_state=42,
    stratify=df["situacao"],
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
modelo_lr = LogisticRegression(max_iter=2000, random_state=42)
modelo_lr.fit(X_train_tfidf, y_train)
pred_lr = modelo_lr.predict(X_test_tfidf)
print("=== Regressão logística ===")
print("Acurácia:", accuracy_score(y_test, pred_lr))
print(classification_report(y_test, pred_lr, digits=3))
print("Matriz de confusão:\n", confusion_matrix(y_test, pred_lr))

In [ ]:
modelo_arvore = DecisionTreeClassifier(max_depth=8, random_state=42)
modelo_arvore.fit(X_train_tfidf, y_train)
pred_arv = modelo_arvore.predict(X_test_tfidf)
print("=== Árvore de decisão ===")
print("Acurácia:", accuracy_score(y_test, pred_arv))
print(classification_report(y_test, pred_arv, digits=3))

In [ ]:
exemplos = [
    "sinto dor no peito intensa e falta de ar mesmo em repouso",
    "exames de rotina normais, sem dor nem falta de ar",
]
X_novo = vectorizer.transform(exemplos)
for texto, r in zip(exemplos, modelo_lr.predict(X_novo)):
    print(f"{r!r}: {texto}")

### Parte 2 — Viés e governança (reflexão)

- Os rótulos seguem uma **regra automática** sobre o CSV da Fase 1, não laudo médico: o modelo aprende essa convenção.
- **Desbalanceamento** entre classes pode inflar acurácia à custa de uma das classes.
- Frases **muito similares** entre registros podem viciar o TF-IDF; em triagem real seriam necessários auditoria, equidade e explicabilidade.

Use este bloco no vídeo da entrega para comentar limitações éticas do protótipo.